# Aula 5 — Conceitos de Séries Temporais

Como usar este notebook: a **Parte A** tem células que o professor roda e
explica durante a aula. Acompanhe na tela, sem precisar digitar nada. A
**Parte B** é com você: complete os exercícios nos lugares marcados com
`# SEU CODIGO AQUI`.

Se ainda não sabe como abrir e salvar sua própria cópia deste notebook,
veja a página **Antes de começar** no material da aula antes de continuar.

## Parte A: Demonstração

### Os dados: 1.096 dias de uma cafeteria

Cada linha é um dia. O `parse_dates` faz o pandas entender que a coluna
`data` é data de verdade, e não texto: é isso que permite ordenar,
filtrar por período e extrair o dia da semana.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Endereço dos dados desta aula no GitHub.
URL_DADOS = "https://raw.githubusercontent.com/klein-natan/nanodegree-AI-Atitus/main/data/vendas_cafeteria.csv"
# Alternativa para testar offline, antes do repositório existir no GitHub:
# URL_DADOS = "../../data/vendas_cafeteria.csv"

dados = pd.read_csv(URL_DADOS, parse_dates=["data"])
dados = dados.sort_values("data").reset_index(drop=True)
dados.head()

In [ ]:
plt.figure(figsize=(11, 4))
plt.plot(dados["data"], dados["vendas"])
plt.xlabel("Data")
plt.ylabel("Vendas do dia (R$)")
plt.title("Três anos de vendas diárias")
plt.show()

print(f"Venda média em 2022: R$ {dados[dados['data'].dt.year == 2022]['vendas'].mean():.2f}")
print(f"Venda média em 2024: R$ {dados[dados['data'].dt.year == 2024]['vendas'].mean():.2f}")

### A média móvel

Em vez do valor de um dia só, a média dos últimos $k$ dias:

$$\text{MM}_k(t) = \frac{1}{k}\sum_{i=0}^{k-1} y_{t-i}$$

Uma janela do tamanho do ciclo apaga aquele ciclo. Como a sazonalidade
principal aqui é semanal, a janela natural é 7.

In [ ]:
# A média móvel suaviza a série: cada ponto vira a média da janela
dados["media_7"] = dados["vendas"].rolling(7).mean()
dados["media_30"] = dados["vendas"].rolling(30).mean()

recorte = dados[dados["data"] >= "2024-01-01"]

plt.figure(figsize=(11, 4))
plt.plot(recorte["data"], recorte["vendas"], label="Vendas do dia")
plt.plot(recorte["data"], recorte["media_7"], label="Média de 7 dias")
plt.plot(recorte["data"], recorte["media_30"], label="Média de 30 dias")
plt.xlabel("Data (2024)")
plt.ylabel("Vendas (R$)")
plt.legend()
plt.show()

### A sazonalidade semanal

`dt.weekday` devolve 0 para segunda e 6 para domingo.

In [ ]:
nomes_dias = ["Segunda", "Terça", "Quarta", "Quinta", "Sexta", "Sábado", "Domingo"]

dados["dia_da_semana"] = dados["data"].dt.weekday
media_por_dia = dados.groupby("dia_da_semana")["vendas"].mean()

for numero, media in media_por_dia.items():
    print(f"{nomes_dias[numero]}: R$ {media:.2f}")

plt.figure(figsize=(8, 4))
plt.bar(nomes_dias, media_por_dia.values)
plt.ylabel("Venda média (R$)")
plt.title("O padrão que se repete toda semana")
plt.show()

### A regra de ouro: o corte é no tempo

Nada de sortear linhas. O treino é tudo até uma data, e o teste é o que
vem depois. Aqui, os últimos 28 dias ficam escondidos.

In [ ]:
DIAS_DE_TESTE = 28

treino = dados.iloc[:-DIAS_DE_TESTE]
teste = dados.iloc[-DIAS_DE_TESTE:].copy()

print(f"Treino: {len(treino)} dias, de {treino['data'].min().date()} a {treino['data'].max().date()}")
print(f"Teste:  {len(teste)} dias, de {teste['data'].min().date()} a {teste['data'].max().date()}")

### Três previsões que não precisam de modelo

As três, escritas como fórmula:

$$\hat{y}_{t+1} = y_t \qquad \hat{y}_{t+1} = \frac{1}{7}\sum_{i=0}^{6} y_{t-i} \qquad \hat{y}_{t+1} = y_{t-6}$$

A primeira repete o último dia, a segunda usa a média dos últimos sete, e
a terceira repete o mesmo dia da semana anterior. Elas são a régua: um
modelo que não vence as três não paga o próprio trabalho.

In [ ]:
# Ingênua: amanhã é igual ao último dia observado
teste["ingenua"] = treino["vendas"].iloc[-1]

# Média móvel: amanhã é a média dos últimos 7 dias observados
teste["media_7_dias"] = treino["vendas"].iloc[-7:].mean()

# Sazonal ingênua: amanhã é igual ao mesmo dia da semana anterior
ultima_semana = treino["vendas"].iloc[-7:].to_numpy()
teste["sazonal"] = np.tile(ultima_semana, 4)[:DIAS_DE_TESTE]

plt.figure(figsize=(11, 4))
plt.plot(teste["data"], teste["vendas"], label="O que aconteceu")
plt.plot(teste["data"], teste["ingenua"], label="Ingênua")
plt.plot(teste["data"], teste["media_7_dias"], label="Média de 7 dias")
plt.plot(teste["data"], teste["sazonal"], label="Sazonal ingênua")
plt.xlabel("Data")
plt.ylabel("Vendas (R$)")
plt.legend()
plt.show()

### Medindo o erro

MAE e RMSE você já conhece. O MAPE mede o erro em porcentagem do valor
real de cada dia:

$$\text{MAPE} = \frac{100}{n}\sum_{t=1}^{n}\left|\frac{y_t - \hat{y}_t}{y_t}\right|$$

In [ ]:
def calcular_metricas(reais, previstos):
    # Tres jeitos de resumir o mesmo erro: em reais, em reais com punicao
    # extra para erro grande, e em porcentagem
    erro = reais - previstos
    mae = np.abs(erro).mean()
    rmse = np.sqrt((erro ** 2).mean())
    mape = (np.abs(erro / reais)).mean() * 100
    return mae, rmse, mape

for nome, coluna in [("Ingênua", "ingenua"),
                     ("Média de 7 dias", "media_7_dias"),
                     ("Sazonal ingênua", "sazonal")]:
    mae, rmse, mape = calcular_metricas(teste["vendas"], teste[coluna])
    print(f"{nome:<18} MAE R$ {mae:6.2f} | RMSE R$ {rmse:6.2f} | MAPE {mape:5.2f}%")

## Parte B: Exercícios

Complete cada exercício no espaço marcado com `# SEU CODIGO AQUI`. Rode a
célula de verificação logo depois para conferir sua resposta.

### Exercício 1: conhecendo a série

Rode a célula e observe: qual foi o melhor dia de vendas dos três anos?
E o pior?

In [ ]:
print(dados["vendas"].describe().round(2))
print()
print("Melhor dia:", dados.loc[dados["vendas"].idxmax(), "data"].date())
print("Pior dia:  ", dados.loc[dados["vendas"].idxmin(), "data"].date())

In [ ]:
if len(dados) == 1096:
    print(f"✅ A série tem {len(dados)} dias, como esperado.")
else:
    print("❌ Confira se você rodou a célula que carrega os dados, no início do notebook.")

### Exercício 2: os feriados

Rode a célula e compare a venda média de um feriado com a de um dia
comum.

In [ ]:
media_por_feriado = dados.groupby("feriado")["vendas"].mean().round(2)
print(media_por_feriado)

In [ ]:
print("Converse com um colega: por que um feriado derruba tanto as vendas desta cafeteria?")

### Exercício 3: uma média móvel de 14 dias

Crie uma coluna `media_14` com a média móvel de 14 dias da coluna
`vendas`. Use `.rolling(14).mean()`.

In [ ]:
# SEU CODIGO AQUI

In [ ]:
print(dados[["data", "vendas", "media_14"]].tail(3))

In [ ]:
if "media_14" in dados.columns and dados["media_14"].isna().sum() == 13:
    print("✅ A média de 14 dias existe, e os 13 primeiros dias ficaram vazios (é o esperado).")
else:
    print("❌ Confira se você usou rolling(14).mean() na coluna vendas.")

### Exercício 4: cortando no tempo

Separe os últimos 56 dias como teste (o dobro da demonstração). Use
`.iloc` com números negativos.

In [ ]:
DIAS_NOVO_TESTE = 56

In [ ]:
# SEU CODIGO AQUI

In [ ]:
print(f"Treino: {len(meu_treino)} dias")
print(f"Teste:  {len(meu_teste)} dias, começando em {meu_teste['data'].min().date()}")

In [ ]:
if len(meu_teste) == 56 and meu_teste["data"].min() > meu_treino["data"].max():
    print("✅ O teste tem 56 dias e vem todo depois do treino. Corte correto.")
else:
    print("❌ O teste precisa vir depois do treino. Confira o sinal no iloc.")

### Exercício 5: a previsão sazonal ingênua

A regra é $\hat{y}_{t+1} = y_{t-6}$: cada dia previsto copia o mesmo dia
da semana anterior. Construa essa previsão para os 56 dias de teste,
repetindo a última semana do treino com `np.tile` (8 vezes).

In [ ]:
ultima_semana_treino = meu_treino["vendas"].iloc[-7:].to_numpy()

In [ ]:
# SEU CODIGO AQUI

In [ ]:
plt.figure(figsize=(11, 4))
plt.plot(meu_teste["data"], meu_teste["vendas"], label="Real")
plt.plot(meu_teste["data"], meu_teste["sazonal"], label="Sazonal ingênua")
plt.legend()
plt.show()

In [ ]:
if len(meu_teste["sazonal"]) == 56 and meu_teste["sazonal"].nunique() <= 7:
    print("✅ A previsão repete os mesmos 7 valores ao longo dos 56 dias.")
else:
    print("❌ Esperava no máximo 7 valores diferentes, repetidos. Confira o np.tile.")

### Exercício 6: medindo o erro

Calcule MAE, RMSE e MAPE da sua previsão sazonal ingênua nos 56 dias,
usando a função `calcular_metricas` já pronta.

In [ ]:
reais = meu_teste["vendas"]
previstos = meu_teste["sazonal"]

In [ ]:
# SEU CODIGO AQUI

In [ ]:
print(f"MAE:  R$ {meu_mae:.2f}")
print(f"RMSE: R$ {meu_rmse:.2f}")
print(f"MAPE: {meu_mape:.2f}%")

In [ ]:
if meu_mape < 15:
    print("✅ Um MAPE abaixo de 15% já é um piso decente para esta série.")
else:
    print("❌ MAPE alto demais. Confira se reais e previstos estão na mesma ordem.")

### Exercício 7: desafio, uma régua melhor

A sazonal ingênua copia **uma** semana só, e por isso carrega todo o
ruído daquela semana. Construa uma previsão melhor: para cada dia da
semana, use a **média das últimas 4 semanas** do treino.

In [ ]:
ultimas_4_semanas = meu_treino.iloc[-28:]

In [ ]:
# SEU CODIGO AQUI

In [ ]:
mae_media, rmse_media, mape_media = calcular_metricas(
    meu_teste["vendas"], meu_teste["media_semanal"]
)
print(f"Sazonal ingênua: MAE R$ {meu_mae:.2f}")
print(f"Média das 4 semanas: MAE R$ {mae_media:.2f}")

In [ ]:
if mae_media < meu_mae:
    print("✅ A média de 4 semanas ficou melhor: menos ruído de uma semana só.")
else:
    print("❌ Ficou pior. Confira se o map usou a coluna dia_da_semana do teste.")

Agora, em texto: em duas ou três frases, explique por que usar a média de
quatro semanas costuma errar menos do que copiar apenas a última semana.
Edite esta célula (duplo clique nela) e escreva sua resposta no lugar
deste parágrafo.